# Entrega 2 — EDA avanzado, conjunto de datos medido y representación de textos

**Proyecto:** detección de sesgo mediático (*framing*) en medios colombianos.
**Corpus:** titulares de 6 medios durante el gobierno Petro (2022-08 → 2026-07).

Este notebook cubre los puntos **1 a 4** de la entrega:

1. Correcciones y modificaciones respecto a la Entrega 1
2. Análisis exploratorio avanzado
3. Conjunto de datos medido
4. Representación de los textos

El código de procesamiento vive en el paquete `news_corpus` (carpeta
`news-retrieval/src/news_corpus/eda/`) y los recursos léxicos en
`news-retrieval/config/`. Aquí se llaman paso a paso para mostrar cada resultado.

**Requisitos:** Docker Desktop abierto, con el corpus restaurado en la base, y
el kernel del notebook apuntando al entorno `.venv` de `news-retrieval`.
Ejecución completa: 10–15 minutos con el corpus completo.

## 0. Configuración

In [1]:
import os
import shutil
import time
import warnings
from collections import Counter
from pathlib import Path

# El proyecto busca .env y config/ en la carpeta actual: se ubica en news-retrieval/.
raiz = Path.cwd()
while not (raiz / "config" / "tipos_contenido.yaml").exists():
    if raiz.parent == raiz:
        raise RuntimeError("Abre este notebook desde la carpeta news-retrieval/ o una subcarpeta.")
    raiz = raiz.parent
os.chdir(raiz)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sqlalchemy import text

from news_corpus.config.settings import REPO_ROOT, get_settings
from news_corpus.db.session import get_engine
from news_corpus.eda import metricas as M
from news_corpus.eda import pipeline as P
from news_corpus.eda.recursos import Recursos
from news_corpus.eda.salidas import NOMBRES
from news_corpus.eda.texto import ClasificadorContenido, ngramas_contenido

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def nombre(m):
    return NOMBRES.get(m, m)


print("Carpeta del proyecto:", raiz)

Carpeta del proyecto: c:\Users\leonk\Documents\media-bias-detectionkl\media-bias-detection\news-retrieval


In [6]:
# ── Parámetros ──────────────────────────────────────────────────────────────
GOBIERNO = "petro"                      # None = sin filtro de gobierno
MEDIOS = P.MEDIOS_COMPARABLES           # los 6 medios con 48/48 meses
SALIDA = REPO_ROOT.parent / "entrega2"  # media-bias-detection/entrega2 (junto al informe)
LIMPIAR_SALIDA = False                   # borra salidas anteriores de entrega2/
SEMILLA = 42
USAR_TRANSFORMERS = False               # sección 4.7 (opcional, ver instrucciones allí)

In [7]:
if LIMPIAR_SALIDA and SALIDA.exists():
    shutil.rmtree(SALIDA)
for sub in ("dataset", "metricas", "figuras", "representaciones"):
    (SALIDA / sub).mkdir(parents=True, exist_ok=True)

TABLAS = {}   # tablas de métricas que se exportan en el punto 3


def guardar_fig(fig, nombre_archivo):
    fig.tight_layout()
    fig.savefig(SALIDA / "figuras" / nombre_archivo, dpi=150, bbox_inches="tight")
    plt.show()


def heatmap(tabla, titulo, archivo, fmt="{:.1f}", cmap="Blues"):
    fig, ax = plt.subplots(figsize=(1.3 * tabla.shape[1] + 4, 0.42 * tabla.shape[0] + 1.6))
    vals = tabla.to_numpy(dtype=float)
    im = ax.imshow(vals, cmap=cmap, aspect="auto")
    ax.set_xticks(range(tabla.shape[1]), tabla.columns, rotation=30, ha="right")
    ax.set_yticks(range(tabla.shape[0]), tabla.index)
    for i in range(tabla.shape[0]):
        for j in range(tabla.shape[1]):
            if not np.isnan(vals[i, j]):
                ax.text(j, i, fmt.format(vals[i, j]), ha="center", va="center", fontsize=7)
    fig.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(titulo, fontsize=10)
    guardar_fig(fig, archivo)


engine = get_engine()
config_dir = get_settings().config_dir
print("Salidas en:", SALIDA)

Salidas en: C:\Users\leonk\Documents\media-bias-detectionkl\media-bias-detection\entrega2


---
## 1. Correcciones y modificaciones respecto a la Entrega 1

| Aspecto | Entrega 1 | Entrega 2 | Razón del cambio |
|---|---|---|---|
| **Corpus** | Curaduría manual: 1 evento, 3 medios *ficticios* (`corpus_ejemplo.json`). El corpus histórico estaba en construcción y no se reportaron cifras. | ~985.000 titulares **reales** de 6 medios, 48 meses, más el piloto 2013–2016 (17.202 artículos). | El ejemplo ficticio servía para probar el prototipo, no para describir medios reales. |
| **Medios** | 10 medios catalogados; el recolector solo soportaba 5 (sitemaps mensuales). | Se programaron 2 recolectores nuevos (feeds Arc e índices de sitemap): 9 medios recolectables. Quedan 6 que cumplen el criterio de cobertura. | El Espectador, Semana y W Radio no alcanzan la ventana (sus feeds solo cubren meses recientes). Criterio: ≥ 90 % de meses completos. |
| **Ventana temporal** | 20 años previstos (2006–2026). | Gobierno Petro: 2022-08 → 2026-07. | Es la única ventana con ≥ 6 medios. Un solo gobierno controla la agenda política como variable de confusión. |
| **Unidad de análisis** | Texto completo del artículo; medidas por oración. | **Titular.** | El cuerpo no es comparable entre medios (en el piloto: 99 % en El Tiempo vs. 15 % en Blu Radio). El titular está disponible para ≥ 99,8 % de las URLs. |
| **Origen del titular** | — | Reconstruido desde la URL (`title_source = slug`); validable con `news-corpus validar-titulares`. | Los sitemaps históricos no incluyen el titular. Se corrigió además un error: los titulares del sitemap se guardaban sin procedencia. |
| **Limpieza** | Normalizar espacios y conservar los términos originales. | Normalización común: minúsculas, sin tildes ni puntuación; tokens de contenido de ≥ 3 letras. | El titular reconstruido ya viene sin tildes: normalizar todo evita que la procedencia parezca una diferencia entre medios. |
| **Tipo de contenido** | — | Clasificación noticia / servicio (lotería, horóscopo, opinión, multimedia) en `config/tipos_contenido.yaml`. | En la primera corrida del EDA, «sorteo», «chance» u «horóscopos» dominaban los términos frecuentes de varios medios. |
| **Medidas textuales** | Previstas sin cifras; estilo evaluado de forma cualitativa por un LLM. | Cuantitativas: longitudes, TTR, MATTR, MTLD, K de Yule, n-gramas, PMI, coocurrencias. | Pendiente explícito de la Entrega 1 («completar con cifras reales»). |
| **Recursos léxicos** | — | Tesauro temático (`topics.yaml`) y ontología de actores (`actores.yaml`). | Permiten describir el corpus por temas y por tipo de actor. |
| **Agrupación por evento** | Campo `evento_id` asignado a mano. | Imposible de asignar a mano a esta escala. En el punto 4 se muestra cómo las representaciones encuentran el mismo hecho en distintos medios. | Prepara la comparación «mismo hecho, distinto medio» del objetivo del proyecto. |

**Estado real de la base de datos:**

In [8]:
with engine.connect() as c:
    estado = pd.read_sql(text('''
        SELECT government_id AS gobierno, source_id AS medio,
               min(published_date) AS desde, max(published_date) AS hasta,
               count(*) AS articulos,
               count(*) FILTER (WHERE title_source = 'slug') AS titulo_desde_url,
               count(*) FILTER (WHERE length(content) >= 500) AS con_cuerpo
        FROM article GROUP BY 1, 2 ORDER BY 1, 5 DESC'''), c)
    meses = pd.read_sql(text('''
        SELECT source_id AS medio,
               count(*) FILTER (WHERE status = 'completed') AS meses_completos,
               count(*) FILTER (WHERE status = 'failed') AS meses_fallidos
        FROM collection_chunk
        WHERE period_start BETWEEN '2022-08-01' AND '2026-07-01'
        GROUP BY 1 ORDER BY 1'''), c)

display(Markdown("**Artículos por gobierno y medio**"))
display(estado)
display(Markdown("**Cobertura de la ventana 2022-08 → 2026-07 (48 meses)**"))
display(meses)

**Artículos por gobierno y medio**

,gobierno,medio,desde,hasta,articulos,titulo_desde_url,con_cuerpo
0,duque,el_tiempo,2022-08-01,2022-08-06,1399,1398,0
1,duque,blu_radio,2022-08-01,2022-08-06,786,786,0
2,duque,la_republica,2022-08-01,2022-08-06,477,477,0
3,duque,noticias_rcn,2022-08-01,2022-08-06,463,463,0
4,duque,noticias_caracol,2022-08-01,2022-08-06,409,409,0
5,duque,cambio,2022-08-01,2022-08-06,98,98,0
6,petro,el_tiempo,2022-08-07,2026-07-31,406760,406313,0
7,petro,blu_radio,2022-08-07,2026-07-31,204000,203738,0
8,petro,noticias_rcn,2022-08-07,2026-07-31,130184,130180,0
9,petro,la_republica,2022-08-07,2026-07-31,106097,105876,0


**Cobertura de la ventana 2022-08 → 2026-07 (48 meses)**

,medio,meses_completos,meses_fallidos
0,blu_radio,48,0
1,cambio,48,0
2,el_espectador,1,47
3,el_tiempo,48,0
4,la_republica,48,0
5,noticias_caracol,48,0
6,noticias_rcn,48,0
7,semana,3,45
8,w_radio,0,48


---
## 2. Análisis exploratorio avanzado

### 2.0 Carga y preprocesamiento

Para cada titular: **normalización** (minúsculas, sin tildes, ñ→n, sin
puntuación) → **tokenización** → **tokens de contenido** (sin stopwords, números
ni tokens de menos de 3 letras) → **temas** (tesauro) → **actores**
(ontología) → **tipo de contenido**.

In [9]:
t0 = time.time()
recursos = Recursos.cargar(config_dir)
clasificador = ClasificadorContenido.cargar(config_dir)
crudo = P.cargar_corpus(engine, GOBIERNO, MEDIOS)
d = P.construir_dataset(crudo, recursos, clasificador)
d["medio_nombre"] = d["source_id"].map(nombre)
del crudo
print(f"{len(d):,} titulares de {d['source_id'].nunique()} medios · "
      f"{d['mes'].min()} → {d['mes'].max()} · {time.time() - t0:.0f} s")

984,590 titulares de 6 medios · 2022-08 → 2026-07 · 112 s


**Ejemplo de la transformación** (una fila por medio):

In [10]:
ej = (d[d["n_tokens_contenido"] >= 5].sample(frac=1, random_state=SEMILLA)
        .groupby("source_id").head(1))
ej[["medio_nombre", "title", "titulo_limpio", "tokens_contenido", "temas", "actores"]]

,medio_nombre,title,titulo_limpio,tokens_contenido,temas,actores
43213,El Tiempo,Tortugas e iguanas que vendian en mercado negro fueron encontradas,tortugas e iguanas que vendian en mercado negro fueron encontradas,"[tortugas, iguanas, vendian, mercado, negro, encontradas]",[],[]
936318,RCN,Miguel angel pinto a indagatoria en la corte suprema por presunto abuso sexual,miguel angel pinto a indagatoria en la corte suprema por presunto abuso sexual,"[miguel, angel, pinto, indagatoria, corte, suprema, presunto, abuso, sexual]",[procesos_judiciales],[corte_suprema]
700749,Blu Radio,Adios a las filas prosperidad social lanza herramienta para recibir pagos directamente en la cuenta so35,adios a las filas prosperidad social lanza herramienta para recibir pagos directamente en la cuenta so35,"[adios, filas, prosperidad, social, lanza, herramienta, recibir, pagos, directamente, cuenta]",[],[]
424890,Caracol,Modelo de onlyfans se alquila como novia y gana miles de dolares al mes no lo hagan gratis chicas webstory...,modelo de onlyfans se alquila como novia y gana miles de dolares al mes no lo hagan gratis chicas webstory...,"[modelo, onlyfans, alquila, novia, gana, miles, dolares, mes, hagan, gratis, chicas, webstory]",[],[]
954224,Cambio,Asombro por primer nacimiento con vida de mono quimerico,asombro por primer nacimiento con vida de mono quimerico,"[asombro, nacimiento, vida, mono, quimerico]",[],[]
745308,La República,Morgan stanley ve auge de fusiones y adquisiciones mientras aumenta la confianza,morgan stanley ve auge de fusiones y adquisiciones mientras aumenta la confianza,"[morgan, stanley, auge, fusiones, adquisiciones, aumenta, confianza]",[],[]


### 2.1 Estudio descriptivo

In [11]:
g = d.groupby("medio_nombre")
desc = pd.DataFrame({
    "titulares": g.size(),
    "meses": g["mes"].nunique(),
    "titulares_por_mes": (g.size() / g["mes"].nunique()).round(0),
    "palabras_media": g["n_tokens"].mean().round(2),
    "palabras_mediana": g["n_tokens"].median(),
    "palabras_contenido_media": g["n_tokens_contenido"].mean().round(2),
    "caracteres_media": g["n_caracteres"].mean().round(1),
    "pct_titulo_desde_url": (g["title_source"].apply(lambda s: (s == "slug").mean()) * 100).round(1),
}).sort_values("titulares", ascending=False)
TABLAS["01_descriptivo_por_medio"] = desc.reset_index()
desc

,titulares,meses,titulares_por_mes,palabras_media,palabras_mediana,palabras_contenido_media,caracteres_media,pct_titulo_desde_url
medio_nombre,,,,,,,,
El Tiempo,406313,48,8465.0,14.10,13.0,7.79,83.7,100.0
Blu Radio,203738,48,4245.0,14.97,15.0,7.62,83.5,100.0
RCN,130180,48,2712.0,12.24,12.0,6.96,72.8,100.0
La República,105876,48,2206.0,11.75,13.0,6.56,70.7,100.0
Caracol,94108,48,1961.0,15.85,16.0,7.79,86.8,100.0
Cambio,44375,48,924.0,11.28,12.0,6.47,69.1,100.0


In [12]:
vol = d.groupby(["mes", "medio_nombre"]).size().unstack(fill_value=0)
vol.index = pd.PeriodIndex(vol.index, freq="M").to_timestamp()
fig, ax = plt.subplots(figsize=(11, 4.2))
for m in vol.columns:
    ax.plot(vol.index, vol[m], lw=1.4, label=m)
ax.set_ylabel("titulares por mes")
ax.set_title("Volumen mensual de titulares por medio")
ax.legend(ncol=3, fontsize=8)
guardar_fig(fig, "fig01_volumen_mensual.png")
TABLAS["01_volumen_mensual"] = vol.reset_index(names="mes")

In [13]:
orden = d.groupby("medio_nombre")["n_tokens"].median().sort_values().index
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].boxplot([d.loc[d["medio_nombre"] == m, "n_tokens"] for m in orden],
                tick_labels=list(orden), showfliers=False)
axes[0].set_ylabel("palabras por titular")
axes[0].set_title("Longitud de los titulares")
axes[0].tick_params(axis="x", rotation=30)
for m in orden:
    s = d.loc[d["medio_nombre"] == m, "n_tokens"].clip(upper=30)
    axes[1].hist(s, bins=range(1, 32), histtype="step", density=True, label=m, lw=1.3)
axes[1].set_xlabel("palabras por titular")
axes[1].set_title("Distribución de longitudes")
axes[1].legend(fontsize=7)
guardar_fig(fig, "fig02_longitud_titulares.png")

**Tipo de contenido.** No todo lo que publica un portal es noticia. Esta
clasificación separa los productos de servicio, que de otro modo dominarían las
frecuencias:

In [14]:
tipos = pd.crosstab(d["medio_nombre"], d["tipo_contenido"])
pct_tipos = (tipos.div(tipos.sum(axis=1), axis=0) * 100).round(2)
TABLAS["01_tipo_contenido_por_medio"] = pct_tipos.reset_index()
display(Markdown("**% de titulares por tipo de contenido**"))
display(pct_tipos)

display(Markdown("**Ejemplos por tipo** (para verificar las reglas de `config/tipos_contenido.yaml`)"))
display(d[d["tipo_contenido"] != "noticia"].sample(frac=1, random_state=SEMILLA)
          .groupby("tipo_contenido").head(3).sort_values("tipo_contenido")
          [["tipo_contenido", "medio_nombre", "title"]])

**% de titulares por tipo de contenido**

tipo_contenido,horoscopo,loteria,multimedia,noticia,opinion
medio_nombre,,,,,
Blu Radio,0.32,9.95,4.11,85.06,0.56
Cambio,0.00,0.09,0.02,99.85,0.04
Caracol,0.01,4.01,2.04,93.92,0.02
El Tiempo,0.62,1.13,0.02,93.42,4.81
La República,0.00,0.02,0.02,98.48,1.48
RCN,0.29,1.51,0.00,97.26,0.94


**Ejemplos por tipo** (para verificar las reglas de `config/tipos_contenido.yaml`)

,tipo_contenido,medio_nombre,title
163671,horoscopo,El Tiempo,Los horoscopos de mhoni vidente para hoy miercoles de marzo
141313,horoscopo,El Tiempo,Los horoscopos de mhoni vidente para hoy de enero
113700,horoscopo,El Tiempo,Los horoscopos de mhoni vidente para hoy de octubre
534352,loteria,Blu Radio,Loteria de boyaca resultados del boyalotto de hoy sabado de abril de so35
678976,loteria,Blu Radio,Loteria del quindio resultados del ultimo sorteo hoy jueves de enero de so35
570150,loteria,Blu Radio,Resultado la caribena chance de hoy lunes de enero de so35
411318,multimedia,Caracol,Quiso tener una sonrisa de revista pero al odontologo se le fue la mano parezco un caballo webstory rs15
526661,multimedia,Blu Radio,Una solucion a las llegadas tarde conozca los moonwalkers los zapatos mas rapidos del mundo webstory so35
534650,multimedia,Blu Radio,La compania vale mas que el dinero reciclador se nego a vender a su perrita por pesos webstory rs15
120626,opinion,El Tiempo,Descordinados caricatura de guerreros


In [15]:
# A partir de aquí el análisis usa solo noticias.
n = d[d["tipo_contenido"] == "noticia"].reset_index(drop=True)
print(f"Noticias: {len(n):,} de {len(d):,} titulares ({len(n) / len(d):.1%}).")

Noticias: 916,450 de 984,590 titulares (93.1%).


### 2.2 Recursos léxicos: tesauro temático y ontología de actores

- **Tesauro temático** (`config/topics.yaml`): 7 temas generales y 35 subtemas,
  cada uno con sus palabras clave.
- **Ontología de actores** (`config/actores.yaml`): 11 tipos de actor y 51
  actores con sus alias (por ejemplo, *petro*, *presidente petro* → Gustavo
  Petro → gobierno nacional).

Ambos se aplican sobre el titular normalizado. Aquí se usan con fines
**descriptivos**: qué temas y qué actores aparecen en cada medio.

In [16]:
cat_temas = P.catalogo_temas(d)["name"]
tot = n.groupby("medio_nombre").size()

tp = n[["medio_nombre", "temas_padre"]].explode("temas_padre").dropna()
pct_padre = (tp.groupby(["temas_padre", "medio_nombre"]).size().unstack(fill_value=0)
               .div(tot, axis=1) * 100)
pct_padre.index = pct_padre.index.map(lambda t: cat_temas.get(t, t))
sin_tema = n.groupby("medio_nombre")["temas"].apply(lambda s: (s.map(len) == 0).mean() * 100)
TABLAS["02_temas_padre_por_medio"] = pd.concat(
    [pct_padre, sin_tema.rename("(ningún tema)").to_frame().T]).round(2).reset_index(names="tema")
display(Markdown("**% de noticias sin ningún tema del tesauro** (deportes, entretenimiento, "
                 "clima y otros asuntos fuera de su alcance)"))
display(sin_tema.round(1).rename("% sin tema").to_frame().T)
heatmap(pct_padre.round(1), "Tesauro temático: % de noticias por tema general",
        "fig03_temas_por_medio.png")

**% de noticias sin ningún tema del tesauro** (deportes, entretenimiento, clima y otros asuntos fuera de su alcance)

medio_nombre,Blu Radio,Cambio,Caracol,El Tiempo,La República,RCN
% sin tema,67.7,59.9,72.2,67.8,73.1,71.3


In [17]:
st = n[["medio_nombre", "temas"]].explode("temas").dropna()
top_sub = (st.groupby(["medio_nombre", "temas"]).size().rename("noticias").reset_index()
             .sort_values(["medio_nombre", "noticias"], ascending=[True, False])
             .groupby("medio_nombre").head(8))
top_sub["subtema"] = top_sub["temas"].map(lambda t: cat_temas.get(t, t))
top_sub["pct"] = (top_sub["noticias"] / top_sub["medio_nombre"].map(tot) * 100).round(2)
TABLAS["02_subtemas_top"] = top_sub
display(Markdown("**Los 8 subtemas más frecuentes por medio** (% de noticias)"))
top_sub.assign(r=top_sub.groupby("medio_nombre").cumcount() + 1).pivot(
    index="r", columns="medio_nombre", values="subtema")

**Los 8 subtemas más frecuentes por medio** (% de noticias)

medio_nombre,Blu Radio,Cambio,Caracol,El Tiempo,La República,RCN
r,,,,,,
1,Gobierno,Gobierno,Gobierno,Gobierno,Gobierno,Salud
2,Salud,Conflicto armado,Salud,Salud,Economía nacional,Gobierno
3,Guerrillas,Salud,Elecciones,Estados Unidos,Recursos naturales,Cambio climático
4,Seguridad ciudadana,Elecciones,Venezuela,Elecciones,Salud,Guerrillas
5,Elecciones,Guerrillas,Seguridad ciudadana,Narcotráfico,Estados Unidos,Venezuela
6,Fiscalía,Fiscalía,Guerrillas,Seguridad ciudadana,Inflación,Estados Unidos
7,Conflicto armado,Procesos judiciales,Estados Unidos,Venezuela,Comercio,Partidos políticos
8,Venezuela,Educación,Recursos naturales,Congreso,Venezuela,Seguridad ciudadana


In [18]:
ta = n[["medio_nombre", "tipos_actor"]].explode("tipos_actor").dropna()
pct_actor = (ta.groupby(["tipos_actor", "medio_nombre"]).size().unstack(fill_value=0)
               .div(tot, axis=1) * 100)
pct_actor.index = pct_actor.index.map(recursos.tipos_actor)
TABLAS["02_tipos_actor_por_medio"] = pct_actor.round(3).reset_index(names="tipo_actor")
heatmap(pct_actor.round(2), "Ontología de actores: % de noticias que mencionan cada tipo",
        "fig04_actores_por_medio.png", fmt="{:.2f}")

aa = n[["medio_nombre", "actores"]].explode("actores").dropna()
top_act = (aa.groupby(["medio_nombre", "actores"]).size().rename("noticias").reset_index()
             .sort_values(["medio_nombre", "noticias"], ascending=[True, False])
             .groupby("medio_nombre").head(8))
top_act["actor"] = top_act["actores"].map(lambda a: recursos.actores[a].nombre)
TABLAS["02_actores_top"] = top_act
display(Markdown("**Los 8 actores más mencionados por medio**"))
top_act.assign(r=top_act.groupby("medio_nombre").cumcount() + 1).pivot(
    index="r", columns="medio_nombre", values="actor")

**Los 8 actores más mencionados por medio**

medio_nombre,Blu Radio,Cambio,Caracol,El Tiempo,La República,RCN
r,,,,,,
1,Gustavo Petro,Gustavo Petro,Gustavo Petro,Estados Unidos,Estados Unidos,Estados Unidos
2,Alcaldes y gobernadores,Estados Unidos,Estados Unidos,Gustavo Petro,Donald Trump,Gustavo Petro
3,Estados Unidos,Ejército y Fuerzas Militares,Policía,Alcaldes y gobernadores,Gustavo Petro,Policía
4,Policía,Ministros (genérico),Alcaldes y gobernadores,Policía,Ministros (genérico),Ministros (genérico)
5,Ministros (genérico),Congreso (genérico),Donald Trump,Donald Trump,Congreso (genérico),Alcaldes y gobernadores
6,Ejército y Fuerzas Militares,Disidencias de las FARC,Ministros (genérico),Congreso (genérico),ONU y organismos multilaterales,Ejército y Fuerzas Militares
7,Congreso (genérico),Alcaldes y gobernadores,Ejército y Fuerzas Militares,Ministros (genérico),EPS y aseguradores de salud,Congreso (genérico)
8,Disidencias de las FARC,Fiscalía,Congreso (genérico),Ejército y Fuerzas Militares,Asobancaria,Donald Trump


### 2.3 Diversidad léxica

TTR y Guiraud dependen del tamaño del texto, así que **todos los medios se
miden con muestras del mismo número de tokens** (5 repeticiones; media ±
desviación). MATTR (ventana de 500; Covington y McFall, 2010) y MTLD (McCarthy
y Jarvis, 2010) son estables frente a la longitud. K de Yule: menor = más diverso.

In [19]:
t0 = time.time()
tam = int(min(200_000, n.groupby("source_id")["n_tokens"].sum().min() * 0.8))
filas = []
for medio, sub in n.groupby("medio_nombre"):
    for rep in range(5):
        filas.append({"medio": medio, **M.diversidad(M.muestra_tokens(sub["tokens"], tam, rep))})
div = pd.DataFrame(filas).groupby("medio")
diversidad = div.mean().round(4).join(div.std().round(4), rsuffix="_sd")
diversidad.insert(0, "tokens_muestra", tam)
TABLAS["02_diversidad_lexica"] = diversidad.reset_index()
print(f"Muestras de {tam:,} tokens por medio · {time.time() - t0:.0f} s")
diversidad[["ttr", "guiraud", "mattr_500", "mattr_500_sd", "mtld", "mtld_sd", "yule_k", "hapax"]]

Muestras de 200,000 tokens por medio · 9 s


,ttr,guiraud,mattr_500,mattr_500_sd,mtld,mtld_sd,yule_k,hapax
medio,,,,,,,,
Blu Radio,0.0893,39.9402,0.5854,0.0003,130.3039,0.8819,117.5930,0.4808
Cambio,0.0978,43.7590,0.6287,0.0006,163.1468,1.3682,119.0248,0.4853
Caracol,0.0874,39.0668,0.5760,0.0008,115.6721,0.6851,129.3855,0.4819
El Tiempo,0.1024,45.7745,0.6251,0.0010,158.0968,1.7155,119.7863,0.4932
La República,0.0935,41.8158,0.6170,0.0004,147.3391,1.6849,138.3425,0.4922
RCN,0.0853,38.1348,0.6235,0.0002,168.2831,0.5957,117.9330,0.4616


In [20]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, (col, titulo) in zip(axes, [("mattr_500", "MATTR (ventana 500)"), ("mtld", "MTLD")]):
    s = diversidad[col].sort_values()
    ax.barh(s.index, s, xerr=diversidad.loc[s.index, f"{col}_sd"], color="#3b6ea5")
    ax.set_xlim(s.min() * 0.95, s.max() * 1.03)
    ax.set_title(titulo)
fig.suptitle(f"Diversidad léxica (muestras de {tam:,} tokens por medio)")
guardar_fig(fig, "fig05_diversidad_lexica.png")

### 2.4 Frecuencias: unigramas, bigramas y trigramas

Los n-gramas se forman sobre la secuencia completa del titular y solo se exige
que no empiecen ni terminen en stopword: así sobrevive «reforma a la salud»
pero no «de la reforma».

In [21]:
t0 = time.time()
CUENTAS = {}
for medio, sub in n.groupby("medio_nombre"):
    CUENTAS[medio] = {
        "unigrama": Counter(t for ts in sub["tokens_contenido"] for t in ts),
        "bigrama": Counter(g for ts in sub["tokens"] for g in ngramas_contenido(ts, 2)),
        "trigrama": Counter(g for ts in sub["tokens"] for g in ngramas_contenido(ts, 3)),
    }
filas = []
for medio, cs in CUENTAS.items():
    for tipo, c in cs.items():
        for rank, (g, f) in enumerate(c.most_common(30), 1):
            filas.append((medio, tipo, rank, g, f, round(f / tot[medio] * 1e4, 1)))
ngr = pd.DataFrame(filas, columns=["medio", "tipo", "rank", "ngrama", "frecuencia", "por_10k"])
TABLAS["02_ngramas_top"] = ngr
print(f"{time.time() - t0:.0f} s")


def tabla_top(tipo, k=15):
    t = ngr[(ngr["tipo"] == tipo) & (ngr["rank"] <= k)]
    return t.pivot(index="rank", columns="medio", values="ngrama")


display(Markdown("**Unigramas más frecuentes**")); display(tabla_top("unigrama"))
display(Markdown("**Bigramas más frecuentes**")); display(tabla_top("bigrama"))
display(Markdown("**Trigramas más frecuentes**")); display(tabla_top("trigrama"))

13 s


**Unigramas más frecuentes**

medio,Blu Radio,Cambio,Caracol,El Tiempo,La República,RCN
rank,,,,,,
1,colombia,colombia,bogota,colombia,millones,colombia
2,bogota,petro,colombia,bogota,colombia,bogota
3,petro,bogota,anos,estados,trump,rcn
4,medellin,gobierno,petro,unidos,ano,noticias
5,antioquia,gustavo,murio,petro,presidente,emision
6,anos,presidente,mujer,anos,gobierno,millonarios
7,gobierno,estados,hombre,presidente,bogota,nacional
8,santander,corte,medellin,gobierno,petroleo,seleccion
9,bucaramanga,unidos,placa,medellin,banco,casa


**Bigramas más frecuentes**

medio,Blu Radio,Cambio,Caracol,El Tiempo,La República,RCN
rank,,,,,,
1,mananas blu,gustavo petro,estados unidos,estados unidos,estados unidos,noticias rcn
2,presidente petro,estados unidos,presidente petro,gustavo petro,petroleo hoy,emision noticias
3,estados unidos,alvaro uribe,gustavo petro,donald trump,donald trump,seleccion colombia
4,seleccion colombia,presidente petro,donald trump,presidente petro,america latina,luis diaz
5,atletico nacional,corte suprema,dolar hoy,seleccion colombia,inteligencia artificial,james rodriguez
6,miguel uribe,presidente gustavo,miguel uribe,presidente gustavo,gustavo petro,estados unidos
7,zona rural,corte constitucional,papa francisco,luis diaz,reino unido,atletico nacional
8,gustavo petro,conflicto armado,redes sociales,liga betplay,banco central,liga betplay
9,camila zuluaga,donald trump,noticias caracol,elecciones presidenciales,salario minimo,famosos colombia


**Trigramas más frecuentes**

medio,Blu Radio,Cambio,Caracol,El Tiempo,La República,RCN
rank,,,,,,
1,clan del golfo,presidente gustavo petro,pico y placa,pico y placa,precios del dolar,emision noticias rcn
2,pico y placa,millones de pesos,horarios y restricciones,presidente gustavo petro,tasas de interes,america de cali
3,valle de aburra,reforma la salud,placa en bogota,minuto a minuto,precio del dolar,precio del dolar
4,quedo en video,clan del golfo,precio del dolar,jornada de votacion,dolar y petroleo,temblor en colombia
5,blu camila zuluaga,precio del oro,placa en medellin,siga la jornada,precio del oro,dolar en colombia
6,valle del cauca,hoy en colombia,placa en bucaramanga,vivo elecciones presidenciales,tasa de interes,clan del golfo
7,sur de bogota,expresidente alvaro uribe,valle de aburra,elecciones presidenciales siga,presidente gustavo petro,presidente gustavo petro
8,mananas blu camila,carlos fernando galan,quedo en video,clan del golfo,banco de bogota,giro de italia
9,fin de semana,metro de bogota,placa en cali,millones de pesos,proyecto de ley,tour de francia


In [22]:
medios_ord = list(CUENTAS)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, m in zip(axes.ravel(), medios_ord):
    t = ngr[(ngr["medio"] == m) & (ngr["tipo"] == "bigrama")].head(12).iloc[::-1]
    ax.barh(t["ngrama"], t["por_10k"], color="#3b6ea5")
    ax.set_title(m)
    ax.set_xlabel("por 10.000 noticias")
for ax in axes.ravel()[len(medios_ord):]:
    ax.axis("off")
fig.suptitle("Bigramas más frecuentes por medio")
guardar_fig(fig, "fig06_bigramas_por_medio.png")

### 2.5 Colocaciones (PMI)

La frecuencia premia combinaciones de palabras comunes. La **información mutua
puntual** (PMI) premia las palabras que aparecen juntas mucho más de lo
esperable por azar: revela expresiones fijas y nombres compuestos del corpus.
Se exige una frecuencia mínima, porque el PMI sobrevalora lo raro.

In [23]:
bi_total = sum((cs["bigrama"] for cs in CUENTAS.values()), Counter())
uni_total = Counter(t for ts in n["tokens"] for t in ts)
pmi = M.pmi_bigramas(bi_total, uni_total, min_frec=max(30, len(n) // 20_000))
pmi = pmi.sort_values("pmi", ascending=False).round(2)
TABLAS["02_colocaciones_pmi"] = pmi.head(200)
pmi.head(25).reset_index(drop=True)

,bigrama,frecuencia,pmi
0,iago falque,47,20.41
1,btg pactual,47,20.38
2,bankman fried,49,20.35
3,tik tok,51,20.30
4,imane khelif,50,20.24
5,werder bremen,48,20.24
6,elkana bohbot,47,20.24
7,disfuncion erectil,45,20.23
8,miley cyrus,53,20.22
9,auroras boreales,47,20.19


### 2.6 Coocurrencias

**Términos foco.** Para cada término, las palabras que aparecen en el mismo
titular más de lo esperable (PMI a nivel de titular, con un mínimo de
coocurrencias). Describe el campo semántico de cada término en el corpus.

In [24]:
FOCOS = ["petro", "paz", "reforma", "eln", "salud", "elecciones"]
cooc = []
for foco in FOCOS:
    c = M.coocurrencias_foco(n["tokens_contenido"], foco, min_frec=max(5, len(n) // 50_000))
    if not c.empty:
        c = c.sort_values("pmi", ascending=False).head(12)
        c.insert(0, "foco", foco)
        cooc.append(c)
cooc = pd.concat(cooc, ignore_index=True).round(2)
TABLAS["02_coocurrencias_foco"] = cooc
cooc.assign(r=cooc.groupby("foco").cumcount() + 1).pivot(index="r", columns="foco", values="termino")

foco,elecciones,eln,paz,petro,reforma,salud
r,,,,,,
1,atipicas,grabe,firmante,alocucion,subcomision,quebrantos
2,legislativas,cerac,narges,gustavo,pensional,asmet
3,presidenciales,garante,mohammadi,desaprobacion,tributaria,savia
4,anticipadas,guerrilla,ocad,retractarse,hundida,subcomision
5,votacion,lafaurie,firmantes,daysuris,agraria,digestiva
6,interpartidistas,dialogos,facilitador,asperger,proposiciones,mental
7,observadores,otty,comisionado,alocuciones,ponentes,contrarreforma
8,regionales,negociadores,gestor,hilsaca,ponencia,reproductiva
9,ultimas,arauca,exparas,blando,sgp,bucal


**Coocurrencia de tipos de actor**: cuántas noticias mencionan a la vez dos tipos de actor.

In [25]:
tipos_ids = list(recursos.tipos_actor)
mat = pd.DataFrame(0, index=tipos_ids, columns=tipos_ids)
for ts in n["tipos_actor"]:
    for a in ts:
        for b in ts:
            mat.loc[a, b] += 1
mat.index = mat.columns = [recursos.tipos_actor[t] for t in tipos_ids]
TABLAS["02_coocurrencia_tipos_actor"] = mat.reset_index(names="tipo_actor")
heatmap(np.log10(mat.replace(0, np.nan)), "Coocurrencia de tipos de actor (log10 de noticias; "
        "diagonal = noticias con ese tipo)", "fig07_coocurrencia_actores.png", fmt="{:.1f}",
        cmap="Purples")

---
## 3. Conjunto de datos medido

El corpus se entrega preprocesado, con una fila por titular, en **Parquet** (un
archivo por medio, todos bajo el límite de 100 MB de GitHub) y una **muestra en
CSV** legible en Excel. Incluye el contenido de servicio, marcado en
`tipo_contenido`, para que cualquier análisis pueda decidir si lo usa.

In [26]:
COLUMNAS = {
    "id": "Identificador del artículo en la base de datos",
    "source_id": "Medio (identificador)",
    "medio": "Medio (nombre)",
    "government_id": "Gobierno en que se publicó",
    "published_date": "Fecha (del sitemap del medio)",
    "mes": "Mes de publicación (AAAA-MM)",
    "url": "URL original",
    "title": "Titular tal como está en la base",
    "title_source": "Procedencia del titular: slug (URL), sitemap o extracted (página)",
    "section": "Sección derivada de la URL, si existe",
    "tipo_contenido": "noticia, loteria, horoscopo, opinion o multimedia",
    "titulo_limpio": "Titular normalizado: minúsculas, sin tildes ni puntuación",
    "tokens": "Todas las palabras del titular limpio",
    "tokens_contenido": "Palabras sin stopwords, números ni tokens de menos de 3 letras",
    "n_caracteres": "Longitud del titular limpio en caracteres",
    "n_tokens": "Número de palabras",
    "n_tokens_contenido": "Número de palabras de contenido",
    "temas": "Subtemas del tesauro (config/topics.yaml)",
    "temas_padre": "Temas generales del tesauro",
    "actores": "Actores de la ontología mencionados (config/actores.yaml)",
    "tipos_actor": "Tipos de los actores mencionados",
}
datos = d[list(COLUMNAS)].copy()
datos.attrs = {}

tamanos = {}
for medio, sub in datos.groupby("source_id"):
    ruta = SALIDA / "dataset" / f"corpus_{medio}.parquet"
    sub.to_parquet(ruta, index=False, compression="zstd")
    tamanos[f"corpus_{medio}.parquet"] = round(ruta.stat().st_size / 1e6, 1)

muestra = datos.sample(frac=1, random_state=SEMILLA).groupby("source_id").head(400)
for c in ("tokens", "tokens_contenido", "temas", "temas_padre", "actores", "tipos_actor"):
    muestra[c] = muestra[c].map(" | ".join)
muestra.to_csv(SALIDA / "dataset" / "muestra.csv", index=False, encoding="utf-8-sig")

dic = pd.DataFrame({"columna": list(COLUMNAS), "descripción": list(COLUMNAS.values()),
                    "tipo": [str(datos[c].dtype) for c in COLUMNAS]})
(SALIDA / "DICCIONARIO_DATOS.md").write_text(
    "# Diccionario de datos — conjunto medido (Entrega 2)\n\n" + dic.to_markdown(index=False)
    + "\n", encoding="utf-8")

for nombre_t, t in TABLAS.items():
    t.to_csv(SALIDA / "metricas" / f"{nombre_t}.csv", index=False, encoding="utf-8-sig")

display(Markdown(f"**{len(datos):,} filas × {datos.shape[1]} columnas**"))
display(pd.Series(tamanos, name="MB").to_frame())
display(Markdown(f"**{len(TABLAS)} tablas de métricas** en `entrega2/metricas/`"))
datos.head(3)

**984,590 filas × 21 columnas**

,MB
corpus_blu_radio.parquet,29.0
corpus_cambio.parquet,6.1
corpus_el_tiempo.parquet,65.4
corpus_la_republica.parquet,14.9
corpus_noticias_caracol.parquet,14.3
corpus_noticias_rcn.parquet,17.7


**12 tablas de métricas** en `entrega2/metricas/`

,id,source_id,medio,government_id,published_date,mes,url,title,title_source,section,tipo_contenido,titulo_limpio,tokens,tokens_contenido,n_caracteres,n_tokens,n_tokens_contenido,temas,temas_padre,actores,tipos_actor
0,17203,el_tiempo,El Tiempo,petro,2022-08-31,2022-08,https://www.eltiempo.com/mundo/europa/el-chernobil-escoces-viven-20-personas-y-un-apartamento-vale-40-mill...,El chernobil escoces viven personas y un apartamento vale millones,slug,mundo,noticia,el chernobil escoces viven personas y un apartamento vale millones,"[el, chernobil, escoces, viven, personas, y, un, apartamento, vale, millones]","[chernobil, escoces, viven, personas, apartamento, vale, millones]",66,10,7,[],[],[],[]
1,17204,el_tiempo,El Tiempo,petro,2022-08-31,2022-08,https://www.eltiempo.com/unidad-investigativa/rodolfo-hernandez-el-hombre-de-los-dolares-con-quien-lo-vinc...,Rodolfo hernandez el hombre de los dolares con quien lo vinculan,slug,unidad-investigativa,noticia,rodolfo hernandez el hombre de los dolares con quien lo vinculan,"[rodolfo, hernandez, el, hombre, de, los, dolares, con, quien, lo, vinculan]","[rodolfo, hernandez, hombre, dolares, vinculan]",64,11,5,[],[],[],[]
2,17205,el_tiempo,El Tiempo,petro,2022-08-31,2022-08,https://www.eltiempo.com/tecnosfera/novedades-tecnologia/andicom-2022-presidente-gustavo-petro-habla-en-el...,Andicom presidente gustavo petro habla en el evento del sector tic,slug,NaN,noticia,andicom presidente gustavo petro habla en el evento del sector tic,"[andicom, presidente, gustavo, petro, habla, en, el, evento, del, sector, tic]","[andicom, presidente, gustavo, petro, habla, evento, sector, tic]",66,11,8,[gobierno],[politica],[petro],[gobierno_nacional]


---
## 4. Representación de los textos

### 4.1 Selección y justificación

Se comparan cuatro representaciones. Cada una responde a una necesidad distinta
del proyecto:

| Técnica | Qué captura | Por qué es pertinente aquí |
|---|---|---|
| **Bolsa de palabras (BoW)** | Qué palabras usa el titular y cuántas veces | Línea base más simple e interpretable. El sesgo por selección léxica (Hamborg et al., 2019) se expresa justamente en *qué* palabras se eligen. |
| **TF-IDF** (unigramas + bigramas) | Palabras **distintivas** del titular frente al corpus | Reduce el peso de lo genérico («gobierno», «colombia») y destaca lo específico («reforma pensional»). Es la base estándar para comparar documentos cortos y sigue siendo interpretable. |
| **Embeddings de palabras (FastText)** entrenados en el corpus | Similitud **semántica**: palabras que se usan en contextos parecidos | Entrenarlo sobre los propios titulares captura el vocabulario colombiano y político del período. Las subpalabras de FastText toleran variantes y palabras raras, útil porque los titulares vienen de la URL. |
| **Embeddings contextuales** (Sentence-BERT multilingüe, opcional) | Significado del titular completo, en contexto | Encuentran el mismo hecho aunque se cuente con otras palabras. Es la base prevista para agrupar coberturas por evento (sección 4.7). |

Los titulares son textos muy cortos (≈ 7–9 palabras de contenido), así que las
representaciones dispersas (BoW, TF-IDF) quedan casi vacías. Esa es la razón para
complementarlas con representaciones densas (LSA, FastText, contextuales).

### 4.2 Bolsa de palabras (BoW)

In [27]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer


def identidad(tokens):
    return tokens


t0 = time.time()
bow = CountVectorizer(analyzer=identidad, min_df=5)
X_bow = bow.fit_transform(n["tokens_contenido"])
dens = X_bow.nnz / (X_bow.shape[0] * X_bow.shape[1])
print(f"Matriz BoW: {X_bow.shape[0]:,} titulares × {X_bow.shape[1]:,} palabras · "
      f"{X_bow.nnz:,} valores no nulos · densidad {dens:.5%} · {time.time() - t0:.0f} s")

# Tres titulares de ejemplo, de medios distintos
EJ = (n[n["n_tokens_contenido"] >= 6].sample(frac=1, random_state=SEMILLA)
        .groupby("source_id").head(1).head(3).index.tolist())
vocab_bow = np.array(bow.get_feature_names_out())


def vector_disperso(X, i, vocab, k=12, dec=0):
    fila = X[i]
    orden = np.argsort(-fila.data)[:k]
    return {vocab[fila.indices[j]]: round(float(fila.data[j]), dec) for j in orden}


ej_bow = pd.DataFrame([{"medio": n.loc[i, "medio_nombre"], "titular": n.loc[i, "title"],
                        "vector BoW (no nulos)": vector_disperso(X_bow, i, vocab_bow)}
                       for i in EJ])
ej_bow

Matriz BoW: 916,450 titulares × 44,204 palabras · 6,789,019 valores no nulos · densidad 0.01676% · 5 s


,medio,titular,vector BoW (no nulos)
0,RCN,Se agiganta el sueno presidente del barcelona da pistas para el regreso de messi,"{'presidente': 1.0, 'regreso': 1.0, 'messi': 1.0, 'sueno': 1.0, 'barcelona': 1.0, 'pistas': 1.0}"
1,El Tiempo,Jennifer lopez borra su contenido y su foto de perfil de instagram,"{'lopez': 1.0, 'perfil': 1.0, 'jennifer': 1.0, 'instagram': 1.0, 'borra': 1.0, 'contenido': 1.0}"
2,Cambio,Las razones por las que pidieron a suecia investigar la compra de los aviones gripen por parte de colombia,"{'colombia': 1.0, 'parte': 1.0, 'suecia': 1.0, 'razones': 1.0, 'investigar': 1.0, 'aviones': 1.0, 'compra'..."


Cada titular es un vector de tantas dimensiones como palabras tiene el
vocabulario, con valor distinto de cero solo en las palabras que contiene. Por
eso la densidad es mínima.

### 4.3 TF-IDF (unigramas + bigramas)

In [28]:
def uni_bi(tokens):
    return tokens + [f"{a} {b}" for a, b in zip(tokens, tokens[1:])]


t0 = time.time()
tfidf = TfidfVectorizer(analyzer=uni_bi, min_df=5, max_df=0.5, sublinear_tf=True)
X_tfidf = tfidf.fit_transform(n["tokens_contenido"])
vocab_tfidf = np.array(tfidf.get_feature_names_out())
print(f"Matriz TF-IDF: {X_tfidf.shape[0]:,} × {X_tfidf.shape[1]:,} términos "
      f"(unigramas + bigramas) · densidad {X_tfidf.nnz / np.prod(X_tfidf.shape):.5%} · "
      f"{time.time() - t0:.0f} s")

ej_tfidf = pd.DataFrame([{"medio": n.loc[i, "medio_nombre"], "titular": n.loc[i, "title"],
                          "vector TF-IDF (pesos más altos)":
                              vector_disperso(X_tfidf, i, vocab_tfidf, k=8, dec=3)}
                         for i in EJ])
display(ej_tfidf)

idf = pd.Series(tfidf.idf_, index=vocab_tfidf)
ilustrativos = [t for t in ["colombia", "gobierno", "bogota", "petro", "reforma", "eln", "salud",
                            "reforma pensional", "paz total", "cese al fuego", "paro camionero",
                            "reforma laboral", "consulta popular"] if t in idf.index]
display(Markdown("**IDF: lo frecuente pesa poco, lo específico pesa mucho** "
                 "(IDF = log((1 + N) / (1 + df)) + 1)"))
display(pd.concat([idf.nsmallest(6), idf[ilustrativos]]).drop_duplicates().sort_values()
          .round(2).rename("IDF").to_frame())

Matriz TF-IDF: 916,450 × 197,921 términos (unigramas + bigramas) · densidad 0.00523% · 17 s


,medio,titular,vector TF-IDF (pesos más altos)
0,RCN,Se agiganta el sueno presidente del barcelona da pistas para el regreso de messi,"{'regreso messi': 0.52, 'presidente barcelona': 0.484, 'pistas': 0.338, 'sueno': 0.302, 'barcelona': 0.3, ..."
1,El Tiempo,Jennifer lopez borra su contenido y su foto de perfil de instagram,"{'perfil instagram': 0.467, 'borra': 0.405, 'jennifer lopez': 0.371, 'jennifer': 0.335, 'perfil': 0.324, '..."
2,Cambio,Las razones por las que pidieron a suecia investigar la compra de los aviones gripen por parte de colombia,"{'investigar compra': 0.402, 'parte colombia': 0.353, 'aviones gripen': 0.314, 'compra aviones': 0.313, 'g..."


**IDF: lo frecuente pesa poco, lo específico pesa mucho** (IDF = log((1 + N) / (1 + df)) + 1)

,IDF
colombia,3.65
bogota,3.84
petro,4.38
anos,4.58
gobierno,4.75
presidente,4.77
salud,5.26
reforma,5.55
eln,6.01
reforma laboral,7.23


**Aplicación: encontrar el mismo hecho en otros medios.** Con TF-IDF, la
similitud del coseno entre titulares permite buscar, para un titular dado, las
coberturas más parecidas de **otros medios** en los días cercanos. Es el paso que
reemplaza el `evento_id` manual de la Entrega 1.

In [29]:
fechas = pd.to_datetime(n["published_date"]).values


def mismo_hecho(i, X, k=6, ventana_dias=2):
    dif = np.abs((fechas - fechas[i]).astype("timedelta64[D]").astype(int))
    cand = np.where((dif <= ventana_dias) & (n["source_id"].values != n.loc[i, "source_id"]))[0]
    if len(cand) == 0:
        return pd.DataFrame()
    sims = (X[cand] @ X[i].T)
    sims = np.asarray(sims.todense()).ravel() if hasattr(sims, "todense") else np.ravel(sims)
    top = cand[np.argsort(-sims)[:k]]
    return pd.DataFrame({"medio": n.loc[top, "medio_nombre"].values,
                         "fecha": n.loc[top, "published_date"].values,
                         "similitud": np.sort(sims)[::-1][:k].round(3),
                         "titular": n.loc[top, "title"].values})


# Titular de consulta: se busca un tema de alta cobertura; si no aparece, uno al azar.
consulta = None
for patron in ["reforma pensional", "reforma a la salud", "cese al fuego", "paro camionero",
               "consulta popular", "reforma laboral"]:
    hits = n.index[n["titulo_limpio"].str.contains(patron) & (n["n_tokens_contenido"] >= 5)]
    if len(hits):
        consulta = int(np.random.default_rng(SEMILLA).choice(hits))
        break
if consulta is None:
    consulta = int(n[n["n_tokens_contenido"] >= 6].sample(1, random_state=SEMILLA).index[0])

display(Markdown(f"**Consulta** ({n.loc[consulta, 'medio_nombre']}, "
                 f"{n.loc[consulta, 'published_date']}): *{n.loc[consulta, 'title']}*"))
sim_tfidf = mismo_hecho(consulta, X_tfidf)
sim_tfidf

**Consulta** (El Tiempo, 2023-10-10 00:00:00): *Minhacienda da aval fiscal a la reforma pensional cuanto costara*

,medio,fecha,similitud,titular
0,RCN,2023-10-10,0.429,Reforma pensional ministerio de hacienda dio el aval fiscal al proyecto en segundo debate
1,Blu Radio,2023-10-10,0.427,Reforma pensional seria completamente financiable hasta el minhacienda rg10
2,La República,2023-10-10,0.383,El minhacienda le dio el aval fiscal a la ponencia para segundo debate de la pensional
3,Caracol,2023-10-10,0.260,Reforma pensional cual seria el impacto fiscal de este proyecto rg10
4,Caracol,2023-10-11,0.249,Reforma pensional minhacienda da luz verde al proyecto pero reconoce que nuevo sistema valdra mas rg10
5,Blu Radio,2023-10-10,0.239,La reforma pensional crea una deuda impagable asofondos rg10


### 4.4 Semántica latente (LSA)

LSA (*Latent Semantic Analysis*) reduce la matriz TF-IDF a 100 dimensiones
densas con SVD truncada. Cada dimensión agrupa palabras que tienden a aparecer
juntas. La proyección t-SNE muestra si la representación organiza los titulares
por tema o por medio.

In [30]:
from sklearn.decomposition import TruncatedSVD
from sklearn.manifold import TSNE

t0 = time.time()
rng = np.random.default_rng(SEMILLA)
idx_fit = rng.choice(X_tfidf.shape[0], size=min(150_000, X_tfidf.shape[0]), replace=False)
svd = TruncatedSVD(n_components=100, random_state=SEMILLA).fit(X_tfidf[idx_fit])
print(f"Varianza explicada por 100 componentes: {svd.explained_variance_ratio_.sum():.1%} · "
      f"{time.time() - t0:.0f} s")

comp = pd.DataFrame({f"componente {k + 1}": vocab_tfidf[np.argsort(-svd.components_[k])[:8]]
                     for k in range(1, 7)})
TABLAS_REP = {"lsa_terminos_por_componente": comp}
display(Markdown("**Términos con más peso en las primeras componentes** (la 1ª refleja la "
                 "frecuencia general y se omite)"))
comp

Varianza explicada por 100 componentes: 5.7% · 21 s


**Términos con más peso en las primeras componentes** (la 1ª refleja la frecuencia general y se omite)

,componente 2,componente 3,componente 4,componente 5,componente 6,componente 7
0,estados,placa,colombia,colombia,bogota,hoy
1,unidos,pico,petro,seleccion,anos,dolar
2,estados unidos,pico placa,presidente,seleccion colombia,hombre,precio
3,colombia,bogota,gustavo,hoy,mujer,bogota
4,petro,placa bogota,gustavo petro,dolar,millones,precio dolar
5,bogota,colombia,seleccion,precio,agua,petroleo
6,presidente,hoy,seleccion colombia,precio dolar,accidente,dolar petroleo
7,pico,medellin,presidente petro,mundial,policia,petroleo hoy


In [31]:
t0 = time.time()
un_tema = n.index[n["temas_padre"].map(len) == 1]
idx_vis = rng.choice(un_tema, size=min(3000, len(un_tema)), replace=False)
Z = TSNE(n_components=2, perplexity=30, init="pca", random_state=SEMILLA).fit_transform(
    svd.transform(X_tfidf[idx_vis]))
vis = n.loc[idx_vis, ["medio_nombre", "temas_padre"]].copy()
vis["tema"] = vis["temas_padre"].str[0].map(lambda t: cat_temas.get(t, t))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, col, titulo in [(axes[0], "tema", "coloreado por tema"),
                        (axes[1], "medio_nombre", "coloreado por medio")]:
    for k, (valor, sub) in enumerate(vis.groupby(col)):
        pos = vis.index.get_indexer(sub.index)
        ax.scatter(Z[pos, 0], Z[pos, 1], s=5, alpha=0.6, label=valor, color=plt.cm.tab10(k))
    ax.set_title(f"t-SNE de LSA ({len(idx_vis):,} titulares), {titulo}")
    ax.legend(markerscale=3, fontsize=7)
    ax.set_xticks([]); ax.set_yticks([])
guardar_fig(fig, "fig08_tsne_lsa.png")
print(f"{time.time() - t0:.0f} s")

6 s


Si los titulares se agrupan por **tema** pero se mezclan por **medio**, la
representación captura de qué se habla, no quién lo publica. Para estudiar
sesgo, eso es lo deseable: se compara *cómo* cuentan los medios un mismo tema.

### 4.5 Embeddings de palabras: FastText entrenado en el corpus

FastText (Bojanowski et al., 2017) aprende un vector denso de 100 dimensiones
por palabra a partir de sus contextos, e incluye subpalabras (n-gramas de
caracteres). Gracias a eso puede representar palabras raras o variantes que no
vio en el entrenamiento. Se usa la variante *skip-gram*, mejor para palabras
poco frecuentes. Un titular se representa como el **promedio de los vectores** de
sus palabras de contenido, menos el vector medio del corpus.

La calidad de los vecinos depende del volumen de entrenamiento. Con poco texto,
el modelo se apoya en la forma de la palabra y aparecen vecinos como
«paz» → «paez». Por eso el número de épocas se ajusta al tamaño del corpus.

In [32]:
from gensim.models import FastText

t0 = time.time()
frases = n["tokens_contenido"].tolist()
# Épocas según el tamaño: ~1 millón de pasadas por titular en total. Con el corpus
# completo (~800.000 noticias) son 5; con un corpus pequeño, más (hasta 30).
EPOCAS = int(np.clip(1_000_000 / len(frases), 5, 30))
ft = FastText(sentences=frases, vector_size=100, window=5, min_count=5, sg=1, epochs=EPOCAS,
              bucket=200_000, workers=4, seed=SEMILLA)
print(f"FastText: {len(ft.wv.key_to_index):,} palabras · 100 dimensiones · {EPOCAS} épocas · "
      f"{time.time() - t0:.0f} s")

consultas_ft = [w for w in ["petro", "paz", "reforma", "eln", "salud", "protesta", "corrupcion",
                            "uribe", "inflacion"] if w in ft.wv.key_to_index]
vecinos = pd.DataFrame({w: [f"{v} ({s:.2f})" for v, s in ft.wv.most_similar(w, topn=8)]
                        for w in consultas_ft})
TABLAS_REP["fasttext_vecinos"] = vecinos
display(Markdown("**Palabras más cercanas** (similitud del coseno)"))
vecinos

FastText: 44,244 palabras · 100 dimensiones · 5 épocas · 91 s


**Palabras más cercanas** (similitud del coseno)

,petro,paz,reforma,eln,salud,protesta,corrupcion,uribe,inflacion
0,gustavo (0.89),dialogos (0.70),reformaba (0.87),disidencias (0.79),secsalud (0.83),protestar (0.90),anticorrupcion (0.88),turbay (0.86),desinflacion (0.92)
1,presidente (0.75),total (0.69),reformar (0.86),disidencia (0.78),gestarsalud (0.80),protesto (0.90),corrupta (0.83),miguel (0.84),estanflacion (0.91)
2,alocucion (0.75),otty (0.68),reformas (0.85),tibu (0.76),minsalud (0.72),protestan (0.88),irrupcion (0.81),alvaro (0.80),hiperinflacion (0.86)
3,gustav (0.74),dialogue (0.67),contrarreforma (0.84),farc (0.75),cafesalud (0.71),protestaba (0.85),corrupto (0.81),otalvaro (0.78),deflacion (0.86)
4,petrovideos (0.71),danilo (0.66),tributaria (0.83),guerrillero (0.75),eps (0.69),protestando (0.85),corruptos (0.78),migue (0.71),inflacionaria (0.81)
5,interlocucion (0.69),dialogar (0.66),reformarse (0.83),guerrilla (0.74),saluda (0.69),protestas (0.85),corruptor (0.78),tarazona (0.69),inflacionario (0.80)
6,escaf (0.68),timochenko (0.65),reformara (0.82),arauca (0.74),talud (0.67),protestaron (0.84),escandalo (0.76),miguelito (0.69),desacelero (0.79)
7,gobierna (0.68),dialogo (0.65),pensional (0.82),clan (0.73),hospitalidad (0.66),protestaban (0.83),escandalos (0.73),uribista (0.68),desaceleracion (0.79)


In [33]:
# Palabras que el modelo nunca vio: FastText las representa por sus subpalabras.
for w in ["petrismo", "reformita", "antipetrista", "desgobierno"]:
    vista = "vista" if w in ft.wv.key_to_index else "NO vista en el entrenamiento"
    print(f"{w:14s} ({vista}) → " + ", ".join(v for v, _ in ft.wv.most_similar(w, topn=5)))

palabra = "petro" if "petro" in ft.wv.key_to_index else ft.wv.index_to_key[0]
print(f"\nVector de '{palabra}' (primeras 10 de 100 dimensiones):")
print(np.round(ft.wv[palabra][:10], 3))

petrismo       (vista) → uribismo, politiquero, politiquera, petrista, uribista
reformita      (NO vista en el entrenamiento) → reformar, reforma, reformara, reformista, reformas
antipetrista   (vista) → petrista, uribista, antipersona, quiroz, antipersonal
desgobierno    (vista) → gobierno, gobierna, gobierne, gobiernan, gobiernos

Vector de 'petro' (primeras 10 de 100 dimensiones):
[ 0.528 -0.049  0.055  0.388 -0.285 -0.471  0.29  -0.399  0.274  0.013]


In [34]:
def vector_titular(tokens):
    vs = [ft.wv[t] for t in tokens if t in ft.wv.key_to_index]
    return np.mean(vs, axis=0) if vs else np.zeros(ft.vector_size)


# El promedio de vectores comparte una componente común (palabras frecuentes) que
# hace que todos los titulares se parezcan. Se resta el vector medio del corpus
# antes de comparar (variante simple de Arora et al., 2017).
idx_media = rng.choice(len(n), size=min(50_000, len(n)), replace=False)
VECTOR_MEDIO = np.mean([vector_titular(n.loc[i, "tokens_contenido"]) for i in idx_media], axis=0)


def vector_centrado(tokens):
    v = vector_titular(tokens) - VECTOR_MEDIO
    return v / (np.linalg.norm(v) + 1e-9)


ej_ft = pd.DataFrame([{"medio": n.loc[i, "medio_nombre"], "titular": n.loc[i, "title"],
                       "vector FastText (primeras 6 de 100 dim.)":
                           [round(float(x), 3)
                            for x in vector_centrado(n.loc[i, "tokens_contenido"])[:6]]}
                      for i in EJ])
display(ej_ft)

# El mismo ejercicio de 4.3 («mismo hecho»), ahora con vectores densos.
dif = np.abs((fechas - fechas[consulta]).astype("timedelta64[D]").astype(int))
cand = np.where((dif <= 2) & (n["source_id"].values != n.loc[consulta, "source_id"]))[0]
V = np.vstack([vector_centrado(n.loc[i, "tokens_contenido"]) for i in cand])
q = vector_centrado(n.loc[consulta, "tokens_contenido"])
s = V @ q
top = np.argsort(-s)[:6]
sim_ft = pd.DataFrame({"medio": n.loc[cand[top], "medio_nombre"].values,
                       "similitud": s[top].round(3),
                       "titular": n.loc[cand[top], "title"].values})
display(Markdown(f"**Mismo hecho con FastText.** Consulta: *{n.loc[consulta, 'title']}*"))
sim_ft

,medio,titular,vector FastText (primeras 6 de 100 dim.)
0,RCN,Se agiganta el sueno presidente del barcelona da pistas para el regreso de messi,"[-0.161, -0.147, -0.088, -0.115, -0.09, -0.004]"
1,El Tiempo,Jennifer lopez borra su contenido y su foto de perfil de instagram,"[-0.089, -0.13, 0.07, 0.171, -0.064, -0.058]"
2,Cambio,Las razones por las que pidieron a suecia investigar la compra de los aviones gripen por parte de colombia,"[0.122, 0.09, -0.042, -0.054, -0.02, -0.094]"


**Mismo hecho con FastText.** Consulta: *Minhacienda da aval fiscal a la reforma pensional cuanto costara*

,medio,similitud,titular
0,Blu Radio,0.883,Reforma pensional seria completamente financiable hasta el minhacienda rg10
1,RCN,0.873,Reforma pensional ministerio de hacienda dio el aval fiscal al proyecto en segundo debate
2,Caracol,0.871,Reforma pensional cual seria el impacto fiscal de este proyecto rg10
3,La República,0.860,El minhacienda le dio el aval fiscal a la ponencia para segundo debate de la pensional
4,Caracol,0.843,Reforma pensional minhacienda da luz verde al proyecto pero reconoce que nuevo sistema valdra mas rg10
5,La República,0.842,Reforma subira pasivo pensional en billones


### 4.6 Comparación de representaciones

In [35]:
comparacion = pd.DataFrame([
    {"representación": "BoW", "dimensiones": X_bow.shape[1],
     "densidad": f"{X_bow.nnz / np.prod(X_bow.shape):.4%}", "interpretable": "sí",
     "captura sinónimos": "no", "uso previsto": "línea base; frecuencias por medio"},
    {"representación": "TF-IDF (1-2 gramas)", "dimensiones": X_tfidf.shape[1],
     "densidad": f"{X_tfidf.nnz / np.prod(X_tfidf.shape):.4%}", "interpretable": "sí",
     "captura sinónimos": "no", "uso previsto": "términos distintivos; búsqueda del mismo hecho"},
    {"representación": "LSA (SVD sobre TF-IDF)", "dimensiones": 100, "densidad": "100%",
     "interpretable": "parcial", "captura sinónimos": "en parte",
     "uso previsto": "visualización; reducción de dimensión"},
    {"representación": "FastText (corpus)", "dimensiones": 100, "densidad": "100%",
     "interpretable": "por vecinos", "captura sinónimos": "sí",
     "uso previsto": "semántica del vocabulario; palabras raras"},
    {"representación": "Sentence-BERT (opcional)", "dimensiones": 384, "densidad": "100%",
     "interpretable": "no", "captura sinónimos": "sí, en contexto",
     "uso previsto": "agrupar coberturas por evento"},
])
TABLAS_REP["comparacion_representaciones"] = comparacion

for nombre_t, t in TABLAS_REP.items():
    t.to_csv(SALIDA / "representaciones" / f"{nombre_t}.csv", index=False, encoding="utf-8-sig")
pd.concat([ej_bow.assign(representacion="BoW").rename(columns={"vector BoW (no nulos)": "vector"}),
           ej_tfidf.assign(representacion="TF-IDF").rename(
               columns={"vector TF-IDF (pesos más altos)": "vector"}),
           ej_ft.assign(representacion="FastText").rename(
               columns={"vector FastText (primeras 6 de 100 dim.)": "vector"})]
          ).to_csv(SALIDA / "representaciones" / "ejemplos_vectores.csv", index=False,
                   encoding="utf-8-sig")
sim_tfidf.assign(metodo="TF-IDF").to_csv(SALIDA / "representaciones" / "mismo_hecho_tfidf.csv",
                                         index=False, encoding="utf-8-sig")
comparacion

,representación,dimensiones,densidad,interpretable,captura sinónimos,uso previsto
0,BoW,44204,0.0168%,sí,no,línea base; frecuencias por medio
1,TF-IDF (1-2 gramas),197921,0.0052%,sí,no,términos distintivos; búsqueda del mismo hecho
2,LSA (SVD sobre TF-IDF),100,100%,parcial,en parte,visualización; reducción de dimensión
3,FastText (corpus),100,100%,por vecinos,sí,semántica del vocabulario; palabras raras
4,Sentence-BERT (opcional),384,100%,no,"sí, en contexto",agrupar coberturas por evento


### 4.7 (Opcional) Embeddings contextuales: Sentence-BERT multilingüe

Requiere instalar `sentence-transformers` (descarga PyTorch, ~1 GB) y descarga
un modelo de ~470 MB la primera vez. Para activarla:

1. En la terminal, **sin** el entorno activado:
   `python -m uv pip install --python .venv/Scripts/python.exe -e ".[contextual]"`
2. Poner `USAR_TRANSFORMERS = True` en la celda de parámetros y volver a ejecutar.

Codifica solo los titulares candidatos de la búsqueda «mismo hecho» (cientos),
no el corpus completo.

In [ ]:
if USAR_TRANSFORMERS:
    from sentence_transformers import SentenceTransformer

    t0 = time.time()
    st_modelo = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    textos = [n.loc[consulta, "title"]] + n.loc[cand, "title"].tolist()
    E = st_modelo.encode(textos, batch_size=64, normalize_embeddings=True,
                         show_progress_bar=False)
    s = E[1:] @ E[0]
    top = np.argsort(-s)[:6]
    sim_ctx = pd.DataFrame({"medio": n.loc[cand[top], "medio_nombre"].values,
                            "similitud": s[top].round(3),
                            "titular": n.loc[cand[top], "title"].values})
    sim_ctx.to_csv(SALIDA / "representaciones" / "mismo_hecho_contextual.csv", index=False,
                   encoding="utf-8-sig")
    print(f"Vector de {E.shape[1]} dimensiones por titular · {time.time() - t0:.0f} s")
    display(sim_ctx)
else:
    print("Sección opcional desactivada (USAR_TRANSFORMERS = False).")

Sección opcional desactivada (USAR_TRANSFORMERS = False).


---
## Archivos generados

In [37]:
for p in sorted(SALIDA.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(SALIDA.parent)}  ({p.stat().st_size / 1e6:.1f} MB)")

entrega2\dataset\corpus_blu_radio.parquet  (29.0 MB)
entrega2\dataset\corpus_cambio.parquet  (6.1 MB)
entrega2\dataset\corpus_el_tiempo.parquet  (65.4 MB)
entrega2\dataset\corpus_la_republica.parquet  (14.9 MB)
entrega2\dataset\corpus_noticias_caracol.parquet  (14.3 MB)
entrega2\dataset\corpus_noticias_rcn.parquet  (17.7 MB)
entrega2\dataset\muestra.csv  (1.4 MB)
entrega2\DICCIONARIO_DATOS.md  (0.0 MB)
entrega2\figuras\fig01_volumen_mensual.png  (0.1 MB)
entrega2\figuras\fig02_longitud_titulares.png  (0.1 MB)
entrega2\figuras\fig03_temas_por_medio.png  (0.1 MB)
entrega2\figuras\fig04_actores_por_medio.png  (0.1 MB)
entrega2\figuras\fig05_diversidad_lexica.png  (0.0 MB)
entrega2\figuras\fig06_bigramas_por_medio.png  (0.2 MB)
entrega2\figuras\fig07_coocurrencia_actores.png  (0.2 MB)
entrega2\figuras\fig08_tsne_lsa.png  (0.4 MB)
entrega2\metadata.json  (0.0 MB)
entrega2\metricas\01_descriptivo_por_medio.csv  (0.0 MB)
entrega2\metricas\01_tipo_contenido_por_medio.csv  (0.0 MB)
entrega2\met

---
### Referencias


- Arora, S., Liang, Y. y Ma, T. (2017). A simple but tough-to-beat baseline for sentence embeddings. *ICLR*.
- Bojanowski, P., Grave, E., Joulin, A. y Mikolov, T. (2017). Enriching word vectors with subword information. *TACL*, 5.
- Covington, M. A. y McFall, J. D. (2010). Cutting the Gordian knot: The moving-average type–token ratio (MATTR). *Journal of Quantitative Linguistics*, 17(2).
- Deerwester, S. et al. (1990). Indexing by latent semantic analysis. *JASIS*, 41(6).
- Hamborg, F., Donnay, K. y Gipp, B. (2019). Automated identification of media bias in news articles: an interdisciplinary literature review. *International Journal on Digital Libraries*, 20(4).
- McCarthy, P. M. y Jarvis, S. (2010). MTLD, vocd-D, and HD-D: A validation study of sophisticated approaches to lexical diversity assessment. *Behavior Research Methods*, 42(2).
- Reimers, N. y Gurevych, I. (2019). Sentence-BERT: Sentence embeddings using Siamese BERT-networks. *EMNLP*.